In [1]:
from sqlalchemy import create_engine, text # banco de dados, várias planilhas de excel alchemy controla isso pelo python
import bcrypt
import dotenv #adicionar variavel de ambiente (endereço do computador e ele olho isso - path psql.exe)
import os

In [2]:
dotenv.load_dotenv("db_palete_caixa.env") # escreve o endereço do banco

True

In [3]:
HOST=os.getenv("HOST")
PORT=os.getenv("PORT")
USER=os.getenv("USER")
DATABASE=os.getenv("DATABASE")
PASS=os.getenv("PASS")
print(HOST, PORT, USER, DATABASE)

localhost 5432 postgres db_palete_caixa


In [4]:
if PASS:
    print(f"{'\u2022'*5}")
else:
    print("Senha não encontrada")

•••••


In [5]:
url = f"postgresql+psycopg2://{USER}:{PASS}@{HOST}:{PORT}/{DATABASE}"

In [6]:
engine = create_engine(
    url, connect_args={"sslmode":"prefer"})

In [7]:
import pandas as pd
df_tabelas = pd.read_sql("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_name IN ('clientes', 'tipos_status', 'tipos_usuarios')
        ORDER BY table_name;
    """, engine)
df_tabelas

,table_name
0,clientes
1,tipos_status
2,tipos_usuarios


In [8]:
#inicio aula 3
criar_tabela = """ 
CREATE TABLE IF NOT EXISTS tipos_status (
    id SERIAL PRIMARY KEY,
    nome_status VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS tipos_usuarios (
     id SERIAL PRIMARY KEY,
     tipo VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS clientes (
    id                  SERIAL PRIMARY KEY,
    cnpj                VARCHAR(14) NOT NULL UNIQUE,
    razao_social        VARCHAR(255) NOT NULL,
    nome_fantasia       VARCHAR(255),
    nome_contato      VARCHAR(255),
    telefone_contato  VARCHAR(20),
    email_contato     VARCHAR(255),
    nome_usuario      VARCHAR(50)  NOT NULL UNIQUE,
    senha_hash        VARCHAR(255) NOT NULL,
    status            VARCHAR(30)  NOT NULL DEFAULT 'ativo',
    tipo              VARCHAR(30)  NOT NULL DEFAULT 'regular',
    data_criacao      TIMESTAMP    NOT NULL DEFAULT NOW(),
    deletado          BOOLEAN      NOT NULL DEFAULT FALSE,
    deletado_em       TIMESTAMP    
    
);

""" 
# liguagem sql: 1ºcoluna: serial 1 a n; 2º coluna: string com 30 caracteres, não ser vazio e repetido
# #""" string que entende o enter doc string ; comando sql exige ponto e virgula no final

with engine.begin() as conn:
    conn.execute(text(criar_tabela))
print("Tabelas criadas com sucesso!!!")

Tabelas criadas com sucesso!!!


In [9]:
inserir_dados = """
INSERT INTO tipos_status (nome_status)
VALUES ('ativo'), ('inativo'), ('suspenso')
ON CONFLICT (nome_status) DO NOTHING;

INSERT INTO tipos_usuarios (tipo)
VALUES ('admin'), ('regular')
ON CONFLICT (tipo) DO NOTHING;

"""

with engine.begin() as conn:
    conn.execute(text(inserir_dados))
print("Dados criadas com sucesso")

Dados criadas com sucesso


In [10]:
df = pd.read_sql(sql="SELECT * FROM tipos_usuarios",
                con=engine)
df

,id,tipo
0,1,admin
1,2,regular


In [11]:
# drop_tabelas = \
#     "DROP TABLE IF EXISTS tipos_status;\n"
    
#     #"DROP TABLE IF EXISTS tabela_inexistente;\n"
#     # "DROP TABLE IF EXISTS clientes;\n"
#     # "DROP TABLE IF EXISTS tipos_usuarios;"

# with engine.begin() as conn:
#     conn.execute(text(drop_tabelas))
# print("Tabela(s) excluídas com sucesso")

In [12]:
# tabela

drop_tabelas = """
DROP TABLE IF EXISTS tabelas_inexistentes;
--DROP TABLE IF EXISTS tipos_status;
--DROP TABLE IF EXISTS tipos_usuarios;
--DROP TABLE IF EXISTS clientes;
"""

# trucate_tabelas = """
# --TRUNCATE TABLE  tabelas_inexistentes;
# --TRUNCATE TABLE  tipos_status;
# --TRUNCATE TABLE  tipos_usuarios;
# --TRUNCATE TABLE  clientes;
# """
with engine.begin() as conn:
     conn.execute(text(drop_tabelas))
print("Tabela(s) excluídas com sucesso")


Tabela(s) excluídas com sucesso


In [13]:
adicionar_coluna = """
ALTER TABLE tipos_status
ADD COLUMN IF NOT EXISTS descricao VARCHAR(255);
"""

with engine.begin() as conn:
    conn.execute(text(adicionar_coluna))

df_tipos_status = pd.read_sql(sql="SELECT*FROM tipos_status",
                            con=engine)
df_tipos_status #read comando de exucucao

,id,nome_status,descricao
0,1,ativo,Usuario liberado
1,2,inativo,Usuario com pendencia de pagamento
2,3,suspenso,Usuario com licença fora do prazo


In [14]:
# adicionar texto na coluna descricao
update_tabela = """
    UPDATE tipos_status
    SET descricao = 'Usuario com licença fora do prazo'
    WHERE nome_status = 'suspenso';
"""

with engine.begin() as conn:
    conn.execute(text(update_tabela))

df_tipos_status = pd.read_sql("SELECT*FROM tipos_status", engine)
df_tipos_status

,id,nome_status,descricao
0,1,ativo,Usuario liberado
1,2,inativo,Usuario com pendencia de pagamento
2,3,suspenso,Usuario com licença fora do prazo


In [27]:
cnpj            = "12345678000190"
razao_social    = "prandiano"
nome_contato   = "Ricieri"
nome_usuario    = "prandiano"
tipo            = "admin"   

In [29]:
#transforma a senha em hash
import getpass
senha=getpass.getpass("Inserir sua senha: ")
confirmar = getpass.getpass("Confirme sua senha: ")
if senha == confirmar:
    senha_hash = bcrypt.hashpw(senha.encode(),
            bcrypt.gensalt()).decode() #transforma a senha em hash
    print("senha coincidem")
else:
    print("Senhas não coincidem")

print(senha_hash)

senha coincidem
$2b$12$3/TqIGQfkQs.LckWTdni/uLdYj9SfU1tYX2tW3Zn4unJRzA8VXrpu


In [30]:
with engine.begin() as conn:
    inserir_usuario = f"""
    INSERT INTO clientes (cnpj, razao_social, nome_contato, nome_usuario,
                            senha_hash, tipo)
    VALUES ('{cnpj}','{razao_social}','{nome_contato}','{nome_usuario}','{senha_hash}','{tipo}')

    """
    conn.execute(text(inserir_usuario))
print("Usuario inserido com sucesso")

Usuario inserido com sucesso


In [31]:
df_clientes = pd.read_sql("SELECT*FROM clientes", engine)
df_clientes

,id,cnpj,razao_social,nome_fantasia,nome_contato,telefone_contato,email_contato,nome_usuario,senha_hash,status,tipo,data_criacao,deletado,deletado_em
0,1,12345678000190,prandiano,None,Ricieri,None,None,prandiano,$2b$12$3/TqIGQfkQs.LckWTdni/uLdYj9SfU1tYX2tW3Z...,ativo,admin,2026-09-27 07:24:52.532807,False,None


In [35]:
from sqlalchemy import Engine
from sqlalchemy import create_engine
from sqlalchemy.exc import IntegrityError

url = f"postgresql+psycopg2://{USER}:{PASS}@{HOST}:{PORT}/{DATABASE}"
def get_engine() -> Engine:
    return create_engine(url, connect_args={"sslmode": "prefer"}, pool_pre_ping=True)

def disconnect_engine(engine: Engine):
    engine.dispose()

In [36]:
def criar_usuario(dados: dict, senha: str):
    """Insere novo usuário. Retorna None em caso de sucesso ou mensagem de erro."""
    senha_hash = bcrypt.hashpw(senha.encode(), bcrypt.gensalt()).decode()
    print("Criando usuário...")
    print(dados)
    try:
        engine = get_engine()
        with engine.begin() as conn:
            conn.execute(
                text(
                    f"""
                    INSERT INTO clientes
                        (cnpj, razao_social, nome_fantasia, nome_contato,
                         telefone_contato, email_contato, nome_usuario, senha_hash)
                    VALUES ('{dados.get('cnpj', None)}', 
                            '{dados.get('razao_social', None)}', 
                            '{dados.get('nome_fantasia', None)}', 
                            '{dados.get('nome_contato', None)}', 
                            '{dados.get('telefone_contato', None)}', 
                            '{dados.get('email_contato', None)}', 
                            '{dados.get('nome_usuario', None)}',
                            '{senha_hash}'
                            )
                    """
                )
            )
        print("Usuário criado com sucesso.")
        disconnect_engine(engine)
        return None
    except IntegrityError as e:
        diag = getattr(e.orig, "diag", None)
        constraint = (diag.constraint_name if diag else "") or ""
        if "cnpj" in constraint:
            return "CNPJ já cadastrado."
        if "nome_usuario" in constraint:
            return "Nome de usuário já em uso."
        mensagem = diag.message_primary if diag else str(e.orig)
        return f"Erro de integridade: {mensagem}"

In [37]:
dados = {
    'cnpj': '55123456000174',
    'razao_social': 'Empresa Exemplo LTDA',
    'nome_contato': 'João Silva',
    'telefone_contato': '11999999999',
    'nome_usuario': 'jsilva'
}
criar_usuario(dados, senha='def')

Criando usuário...
{'cnpj': '55123456000174', 'razao_social': 'Empresa Exemplo LTDA', 'nome_contato': 'João Silva', 'telefone_contato': '11999999999', 'nome_usuario': 'jsilva'}
Usuário criado com sucesso.


In [43]:
# autentifica a senha
def autenticar(nome_usuario: str, senha: str):
    """Retorna dict do usuário se credenciais válidas e status=ativo, senão None."""
    engine = get_engine()
    with engine.connect() as conn:
        row = conn.execute(
            text(
                "SELECT id, nome_usuario, razao_social, status, tipo, senha_hash "
                f"FROM clientes WHERE nome_usuario = '{nome_usuario}' AND deletado = False"
            )).mappings().first()
    if row and bcrypt.checkpw(senha.encode(), row["senha_hash"].encode()):
        if row["status"] == "ativo":
            return dict(row)
        return {"erro": f"Acesso negado — status da conta: {row['status']}"}
    return {"erro": "Credenciais inválidas."}

autenticar(nome_usuario="jsilva", 
           senha="def")

{'id': 2,
 'nome_usuario': 'jsilva',
 'razao_social': 'Empresa Exemplo LTDA',
 'status': 'ativo',
 'tipo': 'regular',
 'senha_hash': '$2b$12$ESFQD92El2BCVk00STYeaOvFylF6/byHgFmN6HnrRtDY90qDMwThS'}

In [44]:
# alterar o Status do cliente
def listar_usuarios():
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(
            text(
                "SELECT id, nome_usuario, razao_social, nome_contato, email_contato, "
                "status, tipo, data_criacao "
                "FROM clientes WHERE deletado = False ORDER BY razao_social"
            )
        ).mappings().all()
    return [dict(r) for r in rows]
listar_usuarios()

[{'id': 2,
  'nome_usuario': 'jsilva',
  'razao_social': 'Empresa Exemplo LTDA',
  'nome_contato': 'João Silva',
  'email_contato': 'None',
  'status': 'ativo',
  'tipo': 'regular',
  'data_criacao': datetime.datetime(2026, 9, 27, 7, 33, 43, 220752)},
 {'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 7, 24, 52, 532807)}]

In [45]:
#deletar usuário, passo o numero do ID
from datetime import datetime
def deletar_usuario(usuario_id: int):
    """Soft-delete: marca deletado=TRUE e registra a data."""
    engine = get_engine()
    with engine.begin() as conn:    # begin() abre transação e faz commit automático ao final
        conn.execute(
            text("UPDATE clientes SET deletado = TRUE, deletado_em = :agora WHERE id = :id"),
            {"agora": datetime.now(), "id": usuario_id},
        )
deletar_usuario(2)  # Exemplo de uso, substitua 1 pelo ID do usuário que deseja deletar


In [46]:
listar_usuarios()

[{'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 7, 24, 52, 532807)}]

In [47]:
#alterar a Senha do usario 1 de senha 123 para def
def alterar_senha(usuario_id: int, nova_senha: str):
    """Atualiza senha do usuário com nova hash."""
    nova_hash = bcrypt.hashpw(nova_senha.encode(), bcrypt.gensalt()).decode()
    engine = get_engine()
    with engine.begin() as conn:
        conn.execute(
            text(f"UPDATE clientes SET senha_hash = '{nova_hash}' WHERE id = {usuario_id}")
        )
alterar_senha(1, "def")

In [48]:
print(autenticar("prandiano", "123"))
print(autenticar("prandiano", "def"))

{'erro': 'Credenciais inválidas.'}
{'id': 1, 'nome_usuario': 'prandiano', 'razao_social': 'prandiano', 'status': 'ativo', 'tipo': 'admin', 'senha_hash': '$2b$12$DEtJ/TP61BGOtT4D2ogAue/sDzJCH0hNrhfzs6g.Kj73kdisResSq'}


In [49]:
#lista os tipos de status, neste caso são 3
def listar_tipos_status() -> list[str]:
    """Retorna lista de nome_status da tabela tipos_status para os dropdowns de status
    na página de alteração de status e criação de usuário."""
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(text("SELECT nome_status FROM tipos_status ORDER BY nome_status")).all()
    return [r[0] for r in rows]
listar_tipos_status()

['ativo', 'inativo', 'suspenso']

In [50]:
#lista os tipos de usuario, neste caso são 2
def listar_tipos_usuario() -> list[str]:
    """Retorna lista de tipo da tabela tipos_usuarios para os dropdowns de tipo na 
    página de criação de usuário."""
    engine = get_engine()
    with engine.connect() as conn:
        rows = conn.execute(text("SELECT tipo FROM tipos_usuarios ORDER BY tipo")).all()
    return [r[0] for r in rows]
listar_tipos_usuario()

['admin', 'regular']

In [63]:
#alterar o status de algum usuário
def alterar_status(usuario_id: int, novo_status: str):
    """Atualiza o status (ativo/inativo) de um usuário."""
    engine = get_engine()
    with engine.begin() as conn:    # begin() abre transação e faz commit automático ao final
        conn.execute(
            text(f"UPDATE clientes SET status = '{novo_status}' WHERE id = {usuario_id}")
        )
alterar_status(1, "inativo")

In [64]:
print(autenticar("prandiano", "deff"))
print(autenticar("prandiano", "def"))
alterar_status(1,"ativo")
print(autenticar("prandiano","def"))

{'erro': 'Credenciais inválidas.'}
{'erro': 'Acesso negado — status da conta: inativo'}
{'id': 1, 'nome_usuario': 'prandiano', 'razao_social': 'prandiano', 'status': 'ativo', 'tipo': 'admin', 'senha_hash': '$2b$12$DEtJ/TP61BGOtT4D2ogAue/sDzJCH0hNrhfzs6g.Kj73kdisResSq'}


In [65]:
listar_usuarios()

[{'id': 1,
  'nome_usuario': 'prandiano',
  'razao_social': 'prandiano',
  'nome_contato': 'Ricieri',
  'email_contato': None,
  'status': 'ativo',
  'tipo': 'admin',
  'data_criacao': datetime.datetime(2026, 9, 27, 7, 24, 52, 532807)}]